# ✅ SafeRoute India — Notebook 04: Validate Model & Routes

End-to-end validation of the trained risk model and routing engine.

**Before running:** complete the full pipeline — `python run_pipeline.py --city chennai`

**Contents:**
1. Model performance metrics (CV scores, feature importance)
2. Graph attribute coverage check
3. Risk score distributions on road edges
4. Time-of-day modifier sensitivity analysis
5. Route comparison: Safest vs Balanced vs Fastest
6. Safety ordering assertions (safest ≤ fastest in risk)
7. Interactive route comparison map

In [ ]:
import sys, os, copy
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import folium
import osmnx as ox
import pickle
from pathlib import Path

from config import TARGET_CITIES, MODEL_FEATURES, RISK_TIERS
from src.routing import (
    load_graph, apply_time_modifiers_to_graph,
    compute_route_summary, find_safe_routes,
    get_segment_risk_colors, get_route_coordinates
)

plt.rcParams['figure.figsize'] = (13, 5)
sns.set_style('darkgrid')
OUT = Path('../outputs')
OUT.mkdir(exist_ok=True)

CITY_KEY = 'chennai'   # ← change as needed
print(f'Validating: {CITY_KEY}')

## 1. Model Performance Metrics

In [ ]:
model_path  = Path('../models/risk_classifier.pkl')
scaler_path = Path('../models/scaler.pkl')

if not model_path.exists():
    print('Model not found. Run: python src/07_train_model.py')
else:
    with open(model_path,  'rb') as f: model  = pickle.load(f)
    with open(scaler_path, 'rb') as f: scaler = pickle.load(f)
    
    print(f'Model type:      {type(model).__name__}')
    print(f'N estimators:    {model.n_estimators}')
    print(f'Feature count:   {model.n_features_in_}')
    print(f'Classes:         {model.classes_}  ({[RISK_TIERS.get(c,c) for c in model.classes_]})')
    
    # Feature importance
    fi_path = Path('../models/feature_importance.csv')
    if fi_path.exists():
        fi = pd.read_csv(fi_path)
        print(f'\nFeature importances:')
        print(fi.to_string(index=False))
    else:
        importances = model.feature_importances_
        fi = pd.DataFrame({'feature': MODEL_FEATURES, 'importance': importances})\
               .sort_values('importance', ascending=False)
        print(f'\nFeature importances:')
        print(fi.to_string(index=False))

In [ ]:
# Feature importance bar chart
if 'fi' in dir():
    fig, ax = plt.subplots(figsize=(9, 4))
    colors = plt.cm.viridis(np.linspace(0.2, 0.85, len(fi)))
    ax.barh(fi['feature'][::-1], fi['importance'][::-1], color=colors, edgecolor='white', lw=0.3)
    ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)', fontsize=11)
    ax.set_title('Random Forest — Feature Importance', fontsize=13, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(str(OUT / 'feature_importance.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 2. Graph Attribute Coverage

In [ ]:
G = load_graph(CITY_KEY)
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()

print(f'Graph: {CITY_KEY}')
print(f'  Nodes: {n_nodes:,}')
print(f'  Edges: {n_edges:,}')

required = ['crime_score','accident_score','flood_score','road_score',
            'composite_risk','effective_weight','predicted_risk_tier']

print(f'\nAttribute coverage:')
print(f'{"Attribute":<30} {"Present":>10} {"Missing":>10} {"Coverage":>10}')
print('─' * 65)
for attr in required:
    present = sum(1 for _,_,d in G.edges(data=True) if attr in d)
    missing = n_edges - present
    pct     = 100 * present / n_edges
    flag    = '✅' if pct >= 99 else ('⚠️' if pct >= 90 else '❌')
    print(f'{flag} {attr:<28} {present:>10,} {missing:>10,} {pct:>9.1f}%')

## 3. Risk Score Distributions on Road Edges

In [ ]:
edge_data = [
    {
        'crime':    float(d.get('crime_score',    0.3)),
        'accident': float(d.get('accident_score', 0.3)),
        'flood':    float(d.get('flood_score',    0.2)),
        'composite':float(d.get('composite_risk', 0.3)),
        'tier':     int(d.get('predicted_risk_tier', 1)),
        'highway':  str(d.get('highway','unknown')),
    }
    for _,_,d in G.edges(data=True)
]
df_edges = pd.DataFrame(edge_data)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
configs = [
    ('crime',     '#ef4444', 'Crime Risk'),
    ('accident',  '#f59e0b', 'Accident Risk'),
    ('flood',     '#3b82f6', 'Flood Risk'),
    ('composite', '#6366f1', 'Composite Risk'),
]
for ax, (col, color, title) in zip(axes, configs):
    vals = df_edges[col]
    ax.hist(vals, bins=50, color=color, alpha=0.8, edgecolor='white', lw=0.2)
    ax.axvline(vals.mean(),   color='black', ls='--', lw=1.5, label=f'μ={vals.mean():.3f}')
    ax.axvline(vals.median(), color='white', ls=':',  lw=1.2, label=f'm={vals.median():.3f}')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Risk Score')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle(f'Edge Risk Score Distributions — {CITY_KEY.title()}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUT / f'edge_risk_distributions_{CITY_KEY}.png'), dpi=150, bbox_inches='tight')
plt.show()

# Risk tier breakdown
tier_counts = df_edges['tier'].value_counts().sort_index()
tier_labels = {0:'LOW', 1:'MEDIUM', 2:'HIGH'}
print('\nRisk Tier Distribution:')
for tier, count in tier_counts.items():
    pct = 100 * count / len(df_edges)
    bar = '█' * int(pct / 2)
    print(f'  {tier_labels.get(tier,tier):<8} {count:>8,} ({pct:>5.1f}%)  {bar}')

## 4. Time-of-Day Modifier Sensitivity

In [ ]:
hours      = list(range(0, 24, 1))
mean_weights = []

for h in hours:
    G_copy = copy.deepcopy(G)
    apply_time_modifiers_to_graph(G_copy, h)
    w = [float(d.get('time_adjusted_weight', 0)) for _,_,d in G_copy.edges(data=True)]
    mean_weights.append(np.mean(w))

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(hours, mean_weights, 'o-', color='#6366f1', lw=2, ms=5, zorder=5)
ax.fill_between(hours, mean_weights, min(mean_weights), alpha=0.15, color='#6366f1')

# Shade risk zones
ax.axvspan(0,  6,  alpha=0.10, color='red',    label='High crime (0–6 AM)')
ax.axvspan(6,  9,  alpha=0.06, color='orange', label='Morning rush (6–9 AM)')
ax.axvspan(18, 21, alpha=0.06, color='orange', label='Evening rush (6–9 PM)')
ax.axvspan(21, 24, alpha=0.08, color='red',    label='Night risk (9 PM–midnight)')

ax.set_xlabel('Hour of Day', fontsize=11)
ax.set_ylabel('Mean Effective Edge Weight', fontsize=11)
ax.set_title(f'Time-of-Day Risk Sensitivity — {CITY_KEY.title()}', fontsize=13, fontweight='bold')
ax.set_xticks(hours)
ax.set_xticklabels([f'{h:02d}:00' for h in hours], rotation=60, ha='right', fontsize=8)
ax.legend(fontsize=8, ncol=2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(str(OUT / f'time_sensitivity_{CITY_KEY}.png'), dpi=150, bbox_inches='tight')
plt.show()

night_mean = np.mean([mean_weights[h] for h in range(0, 6)])
day_mean   = np.mean([mean_weights[h] for h in range(9, 18)])
print(f'Night mean weight (0–6 AM): {night_mean:,.0f}')
print(f'Day mean weight  (9 AM–6 PM): {day_mean:,.0f}')
print(f'Night/Day ratio: {night_mean/day_mean:.2f}×  (expected > 1.2)')

## 5. Route Comparison — Safest vs Balanced vs Fastest

In [ ]:
TEST_ROUTES = {
    'chennai':   ('Chennai Central Railway Station', 'T. Nagar, Chennai'),
    'mumbai':    ('Chhatrapati Shivaji Terminus, Mumbai', 'Bandra West, Mumbai'),
    'delhi':     ('Connaught Place, Delhi', 'Hauz Khas, Delhi'),
    'bengaluru': ('Kempegowda Bus Station, Bengaluru', 'Koramangala, Bengaluru'),
    'hyderabad': ('Secunderabad Railway Station', 'Hitech City, Hyderabad'),
}

origin, destination = TEST_ROUTES.get(CITY_KEY, ('', ''))
TRAVEL_HOUR = 21   # 9 PM — elevated risk

print(f'Computing routes at {TRAVEL_HOUR:02d}:00...')
print(f'  From: {origin}')
print(f'  To:   {destination}\n')

try:
    routes, orig_coords, dest_coords = find_safe_routes(
        origin_address=origin,
        destination_address=destination,
        city_key=CITY_KEY,
        travel_hour=TRAVEL_HOUR,
        include_segments=True,
    )
    print('Routes computed successfully.')
    ROUTES_OK = True
except Exception as e:
    print(f'Routing error: {e}')
    print('Check that the graph is built and the addresses are geocodeable.')
    ROUTES_OK = False

In [ ]:
# Summary table
if ROUTES_OK:
    print(f'{'Route':<12} {'Dist (km)':>10} {'Safety%':>9} {'Avg Risk':>9} {'Max Risk':>9} {'High Segs':>10} {'Label'}')
    print('─' * 75)
    for name, route in routes.items():
        if not route:
            print(f'{name.upper():<12}  — no path found')
            continue
        s = route['summary']
        icon = {'safest':'🟢','balanced':'🟡','fastest':'🔴'}[name]
        print(f'{icon} {name.upper():<10} '
              f'{s["distance_km"]:>10.2f} '
              f'{s["safety_pct"]:>9.1f} '
              f'{s["avg_risk_score"]:>9.3f} '
              f'{s["max_risk_score"]:>9.3f} '
              f'{s["high_risk_segments"]:>5}/{s["total_segments"]:<4} '
              f'{s["risk_label"]}')

## 6. Safety Ordering Assertions

In [ ]:
if ROUTES_OK:
    safest  = routes.get('safest')
    fastest = routes.get('fastest')
    balanced = routes.get('balanced')
    
    print('Safety ordering assertions:')
    
    if safest and fastest:
        s_risk = safest['summary']['avg_risk_score']
        f_risk = fastest['summary']['avg_risk_score']
        s_dist = safest['summary']['distance_km']
        f_dist = fastest['summary']['distance_km']
        
        ok1 = s_risk <= f_risk + 0.01
        print(f'  {'✅' if ok1 else '❌'} safest_risk ({s_risk:.3f}) ≤ fastest_risk ({f_risk:.3f})')
        
        ok2 = f_dist <= s_dist + 0.5
        print(f'  {'✅' if ok2 else '⚠️'} fastest_dist ({f_dist:.2f} km) ≤ safest_dist ({s_dist:.2f} km)')
        
        dist_cost = s_dist - f_dist
        risk_save = f_risk - s_risk
        print(f'\n  Safety trade-off summary:')
        print(f'    Extra distance for safest route: {dist_cost:+.2f} km')
        print(f'    Risk reduction:                 {risk_save:+.3f} ({risk_save*100:+.1f}pp)')
    
    if balanced and safest and fastest:
        b_risk = balanced['summary']['avg_risk_score']
        b_dist = balanced['summary']['distance_km']
        ok3 = safest['summary']['avg_risk_score'] <= b_risk <= fastest['summary']['avg_risk_score'] + 0.05
        print(f'\n  {'✅' if ok3 else '⚠️'} balanced risk ({b_risk:.3f}) is between safest and fastest')

## 7. Interactive Route Comparison Map

In [ ]:
if ROUTES_OK:
    city_center = TARGET_CITIES[CITY_KEY]['center']
    m = folium.Map(location=list(city_center), zoom_start=13, tiles='CartoDB dark_matter')
    
    COLORS = {'safest': '#22c55e', 'balanced': '#f59e0b', 'fastest': '#ef4444'}
    WEIGHTS = {'safest': 7, 'balanced': 5, 'fastest': 4}
    OPACITIES = {'safest': 0.9, 'balanced': 0.75, 'fastest': 0.6}
    
    for name, route in routes.items():
        if not route:
            continue
        fg = folium.FeatureGroup(name=f"{name.title()} Route")
        
        segments = route.get('segments', [])
        if segments:
            for seg in segments:
                folium.PolyLine(
                    seg['coords'],
                    color=seg['color'],
                    weight=WEIGHTS[name],
                    opacity=OPACITIES[name],
                    tooltip=f"{name} | risk={seg['risk']:.2f}"
                ).add_to(fg)
        else:
            folium.PolyLine(
                route['coordinates'],
                color=COLORS[name],
                weight=WEIGHTS[name],
                opacity=OPACITIES[name]
            ).add_to(fg)
        
        fg.add_to(m)
        
        # Summary badge at midpoint
        s   = route['summary']
        mid = route['coordinates'][len(route['coordinates'])//2]
        folium.Marker(
            mid,
            icon=folium.DivIcon(html=(
                f"<div style='background:rgba(0,0,0,0.85);color:{COLORS[name]};"
                f"padding:4px 8px;border-radius:6px;font-size:11px;"
                f"font-family:Arial;white-space:nowrap;border:1px solid {COLORS[name]}'>"
                f"<b>{name.upper()}</b><br>"
                f"{s['distance_km']} km · safety {s['safety_pct']}%"
                f"</div>"
            ))
        ).add_to(m)
    
    # Origin / Dest markers
    folium.Marker(list(orig_coords), popup='Origin',
                  icon=folium.Icon(color='blue', icon='play')).add_to(m)
    folium.Marker(list(dest_coords), popup='Destination',
                  icon=folium.Icon(color='red',  icon='stop')).add_to(m)
    
    folium.LayerControl(collapsed=False).add_to(m)
    
    out_path = str(OUT / f'route_validation_{CITY_KEY}.html')
    m.save(out_path)
    print(f'Validation map saved → {out_path}')
    m

## ✅ Validation Complete

If all assertions passed:
- The risk model assigns meaningful scores to road edges
- Night routing is correctly penalised vs daytime
- Safest route has ≤ risk than fastest route
- Balanced route sits between safest and fastest

**Next step:** run the API and open the frontend map:
```bash
uvicorn api.main:app --host 0.0.0.0 --port 8000
# Open: http://localhost:8000
```